# WildfireSpreadTS -- single-session download + preprocess + train

Combines everything into one Kaggle session (previous diagnostic run proved
the download/format/repo-structure all work, but /kaggle/temp is wiped
between sessions, so this run redoes it end-to-end in one go).

Uses the WildfireSpreadTS authors' OWN train.py + PyTorch Lightning configs
(not a hand-written loader against their Dataset class) -- this is the
fastest reliable path to a real, working training run on real data.

Stages:
1. Clone repo, install deps
2. Download + checksum-verify + extract WildfireSpreadTS.zip (48.4GB) into `/kaggle/temp` (ephemeral, not the 20GB-capped `/kaggle/working`)
3. Preprocess raw GeoTIFFs -> HDF5 via their `CreateHDF5Dataset.py` (also in `/kaggle/temp`)
4. Train via their `train.py` with the README's documented example config, output checkpoints to `/kaggle/working` (small, persists)

This produces a SEPARATE model from our synthetic `WildfireConvLSTM` --
a real U-Net baseline trained on real data, for comparison. It does not
touch or overwrite `models/convlstm/synthetic_baseline_v1`.

In [ ]:
import os, time, hashlib, urllib.request

print('--- disk at start ---', flush=True)
os.system('df -h /kaggle/working /kaggle/temp 2>&1 || df -h /kaggle')

TEMP = '/kaggle/temp/wfts'
os.makedirs(TEMP, exist_ok=True)
OUT = '/kaggle/working/wfts_train_output'
os.makedirs(OUT, exist_ok=True)
print('Scratch:', TEMP, ' Persisted output:', OUT, flush=True)

In [ ]:
# Stage 1: clone repo + install deps
!git clone --depth 1 https://github.com/SebastianGer/WildfireSpreadTS.git /kaggle/working/wfts_repo
!pip install -q -r /kaggle/working/wfts_repo/requirements.txt 2>&1 | tail -20

In [ ]:
# Stage 2: download (chunked with flushed progress logging -- wget's carriage-
# return progress bar does not show up reliably in Kaggle's captured logs)
url = 'https://zenodo.org/records/8006177/files/WildfireSpreadTS.zip?download=1'
expected_md5 = 'dc1a04e63ccc70037b277d585b8fe761'
dest = f'{TEMP}/WildfireSpreadTS.zip'

t0 = time.time()
chunk_size = 1 << 20
report_every = 1000  # MB, less chatty than the diagnostic run
downloaded = 0

with urllib.request.urlopen(url) as resp, open(dest, 'wb') as f:
    total = int(resp.headers.get('Content-Length', 0))
    print(f'Total size: {total / 1e9:.2f} GB', flush=True)
    last_report_mb = 0
    while True:
        chunk = resp.read(chunk_size)
        if not chunk:
            break
        f.write(chunk)
        downloaded += len(chunk)
        mb = downloaded // (1 << 20)
        if mb - last_report_mb >= report_every:
            elapsed = time.time() - t0
            rate = downloaded / elapsed / 1e6 if elapsed > 0 else 0
            pct = 100 * downloaded / total if total else 0
            print(f'  {mb} MB ({pct:.1f}%), {rate:.1f} MB/s, {elapsed:.0f}s elapsed', flush=True)
            last_report_mb = mb

print(f'Download complete: {downloaded / 1e9:.2f} GB in {time.time() - t0:.0f}s', flush=True)

h = hashlib.md5()
with open(dest, 'rb') as f:
    for chunk in iter(lambda: f.read(1 << 20), b''):
        h.update(chunk)
actual = h.hexdigest()
print('expected:', expected_md5, ' actual:', actual, flush=True)
assert actual == expected_md5, 'Checksum mismatch.'
print('OK: checksum verified.', flush=True)

In [ ]:
# Stage 2b: extract
extract_dir = f'{TEMP}/extracted'
os.makedirs(extract_dir, exist_ok=True)
t0 = time.time()
rc = os.system(f'unzip -q {dest} -d {extract_dir}')
print(f'unzip exit code: {rc}, {time.time()-t0:.0f}s', flush=True)
os.system(f'du -sh {extract_dir}')
# Free the zip now that it is extracted -- we no longer need it and it is
# taking scratch space we may want for the HDF5 conversion output.
os.remove(dest)
print('Removed zip after extraction to free scratch space.', flush=True)

In [ ]:
# Stage 3: preprocess raw GeoTIFFs -> HDF5 using the authors' own script
hdf5_dir = f'{TEMP}/hdf5'
os.makedirs(hdf5_dir, exist_ok=True)
t0 = time.time()
rc = os.system(
    f'python /kaggle/working/wfts_repo/src/preprocess/CreateHDF5Dataset.py '
    f'--data_dir {extract_dir} --target_dir {hdf5_dir}'
)
print(f'CreateHDF5Dataset exit code: {rc}, {time.time()-t0:.0f}s', flush=True)
os.system(f'du -sh {hdf5_dir}')
os.system(f'find {hdf5_dir} -maxdepth 2 | head -20')

In [ ]:
# Stage 4: train using the authors' own train.py + documented example config.
# Reduced max_epochs from the README's 200 to a small number first --
# validates the whole pipeline actually works end-to-end before committing
# the rest of the session budget to a long run. Bump this up in a follow-up
# run once this is confirmed working.
os.chdir('/kaggle/working/wfts_repo')
rc = os.system(
    'python3 train.py '
    '--config=cfgs/unet/res18_monotemporal.yaml '
    '--trainer=cfgs/trainer_single_gpu.yaml '
    '--data=cfgs/data_monotemporal_full_features.yaml '
    '--seed_everything=0 '
    '--trainer.max_epochs=5 '
    '--do_test=True '
    f'--data.data_dir {hdf5_dir} '
    f'--trainer.default_root_dir {OUT}'
)
print('train.py exit code:', rc, flush=True)

In [ ]:
# Report what got produced
os.system(f'find {OUT} -iname "*.ckpt" -o -iname "*.yaml" | head -20')
os.system(f'du -sh {OUT}')